# Intrinsic Camera Calibration

Estimates a camera's intrinsic parameters (focal lengths, principal point,
and lens distortion) from photos of a printed chessboard
(`intrinsic_calibration_pattern.pdf`, 9 x 6 inner corners, 24.5 mm squares)
using OpenCV, and prints them in the `intrinsics` format of the POSEIDON
site configs (`poseidon_deploy/configs/sites/<site>.yaml`).

Run this notebook from the `poseidon_deploy/calibration` folder; the example
photos are in `example_intrinsic_cal_imgs/`.

In [ ]:
import glob
import os

import cv2 as cv
import numpy as np

## Detect chessboard corners

Each photo is searched for the chessboard's inner corners, which are then
refined to sub-pixel accuracy. Photos where the board is not found are
skipped.

In [ ]:
PATTERN = (9, 6)  # inner corners per row and column
SQUARE_SIZE_MM = 24.5
IMAGE_GLOB = "example_intrinsic_cal_imgs/*.jpg"

# Stop corner refinement after 30 iterations or 0.001 px movement
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
find_flags = (cv.CALIB_CB_ADAPTIVE_THRESH + cv.CALIB_CB_NORMALIZE_IMAGE
              + cv.CALIB_CB_FAST_CHECK)

# Board corner coordinates in the board plane: (0, 0, 0), (1, 0, 0), ...
board = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
board[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
board *= SQUARE_SIZE_MM

objpoints = []  # 3-D board points for each usable photo
imgpoints = []  # matching 2-D image points
images = sorted(glob.glob(IMAGE_GLOB))

for i, fname in enumerate(images, start=1):
    name = os.path.basename(fname)
    img = cv.imread(fname)
    if img is None:
        print(f"[{i}/{len(images)}] SKIP: could not read {name}")
        continue
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    found, corners = cv.findChessboardCorners(gray, PATTERN, find_flags)
    if not found:
        print(f"[{i}/{len(images)}] FAIL: no chessboard in {name}")
        continue
    objpoints.append(board)
    imgpoints.append(cv.cornerSubPix(gray, corners, (11, 11), (-1, -1),
                                     criteria))
    print(f"[{i}/{len(images)}] OK: {name}")

image_size = gray.shape[::-1]  # (width, height)
print(f"\nUsable photos: {len(objpoints)} of {len(images)}")

## Calibrate and print the intrinsics

OpenCV returns distortion as `[k1, k2, p1, p2, k3]`; POSEIDON (CIRN) orders
it as radial `k1, k2, k3` followed by tangential `p1, p2`.

In [ ]:
if not objpoints:
    raise RuntimeError("No usable calibration photos found.")

rms, mtx, dist, rvecs, tvecs = cv.calibrateCamera(
    objpoints, imgpoints, image_size, None, None)
k1, k2, p1, p2, k3 = dist.ravel()[:5]
print(f"RMS re-projection error: {rms:.4f} px\n")

width, height = image_size
intrinsics = [
    (width, "number of pixel columns (NU)"),
    (height, "number of pixel rows (NV)"),
    (mtx[0, 2], "principal point U (c0U)"),
    (mtx[1, 2], "principal point V (c0V)"),
    (mtx[0, 0], "focal length U (fx)"),
    (mtx[1, 1], "focal length V (fy)"),
    (k1, "radial distortion d1"),
    (k2, "radial distortion d2"),
    (k3, "radial distortion d3"),
    (p1, "tangential distortion t1"),
    (p2, "tangential distortion t2"),
]

# Paste into the site YAML
print("intrinsics:")
for value, label in intrinsics:
    text = f"{int(value)}" if label.startswith("number") else f"{value:.4f}"
    print(f"  - {text:<11} # {label}")

## Check the re-projection error

The mean distance (pixels) between detected corners and corners projected
with the fitted model; values well under one pixel indicate a good fit.

In [ ]:
errors = []
for board_pts, image_pts, rvec, tvec in zip(objpoints, imgpoints, rvecs,
                                            tvecs):
    projected, _ = cv.projectPoints(board_pts, rvec, tvec, mtx, dist)
    # L2 norm of all corner offsets, as cv.norm(..., NORM_L2); NumPy
    # avoids OpenCV's type checks and its version-dependent shapes
    offsets = image_pts.reshape(-1, 2) - projected.reshape(-1, 2)
    errors.append(np.sqrt(np.sum(offsets**2)) / len(projected))
print(f"Mean re-projection error: {np.mean(errors):.4f} px")